# M3 實驗 Arm C:混合訓練(EPFL + Roboflow 一起訓)

回答:「**把 Roboflow 混進 EPFL 一起訓**,會不會比分階段(Arm B)或純 EPFL(Arm A)好?」

| | Arm A(已跑) | Arm B(已跑) | **Arm C(本筆記本)** |
|---|---|---|---|
| 訓練資料 | EPFL 540 | Roboflow 40ep → EPFL 60ep | **EPFL train + Roboflow train 合併,一次訓** |
| train 圖數 | 360 | 360(+100 在 stage1) | **460** |
| train 框數 | 3394 | 3394(+160 在 stage1) | **3554** |
| 訓練設定 | 60ep / 704 / bs4 / ga4 / lr1e-4 | stage2 同左 | **同左** |

**A、B 不重訓**,直接對照 `docs/M3_實驗_分階段vsEPFL_20260810.md` 記錄的數字(評估函式與 test 集完全相同)。

### 設計要點
- **valid / test 維持純 EPFL** —— 混進 Roboflow 會讓分數失去對照意義。
- 用**同一個 `data.zip`**(內含 `epfl/` 與 `rf/`),在 Colab 裡就地合併,保證資料與 A/B 一致。
- ⚠ **已知風險**:Roboflow 100 張中 79 張只有 1 個框,人工抽查發現嚴重漏標/錯標
  (`knife32` 一個框包住 4 把刀+砧板、佔全圖 49.6%;`kitchenwoodcounter27` 畫面中央的砧板+食材完全沒標)。
  DETR 的 Hungarian matching 會把沒配到 GT 的 query 推向 no-object → **這些圖會教模型「砧板/食材是背景」**。
  Arm B 因為 stage2 只吃乾淨 EPFL 而被保護,Arm C 沒有。若 C 掉分,把下面的 `CLEAN_RF` 打開再跑一次即可定位原因。

**用法**:GPU(T4)→ 依序執行 → 上傳 `data.zip`。約 1~1.5 小時(只訓一段)。
⚠ 含 EPFL(CC-NC)→ 驗證用、不出貨。

In [ ]:
# 1) 安裝
!nvidia-smi -L
!pip -q install "rfdetr[train,loggers]" supervision

In [ ]:
# 2) 上傳 data.zip(與 Arm A/B 同一個檔,內含 epfl/ 與 rf/)
from google.colab import files
print('請上傳 data.zip(本機路徑:data/m3_finetune_r2/data.zip)')
up = files.upload()
!unzip -q -o data.zip -d /content
!echo EPFL: && ls /content/epfl && echo RF: && ls /content/rf

In [ ]:
# 3) 建立混合訓練集:train = EPFL + Roboflow,valid/test 維持純 EPFL
import json, shutil, os
from collections import Counter

# False = 原始混合(與 Arm B 用同一份 rf 資料,apples-to-apples)
# True  = 先剔除漏標/錯標嚴重的 Roboflow 圖(診斷用,C 掉分時才開)
CLEAN_RF = False

ANN = '_annotations.coco.json'
SCENE_PREFIXES = ('kitchenwoodcounter', 'kitchenstonecounter', 'ussink')  # 場景照,漏標嚴重
BLOB_AREA_RATIO = 0.40   # 單框佔全圖 >40% → 特寫,或多物件被合併成一個大框

def _load(split_dir):
    d = json.load(open(os.path.join(split_dir, ANN), encoding='utf-8'))
    by = {im['id']: {'image': im, 'anns': []} for im in d['images']}
    for a in d['annotations']:
        if a['image_id'] in by:
            by[a['image_id']]['anns'].append(a)
    return d['categories'], list(by.values())

def _filter_rf(records):
    kept, dropped = [], Counter()
    for r in records:
        if r['image']['file_name'].startswith(SCENE_PREFIXES):
            dropped['場景照漏標'] += 1; continue
        area = r['image']['width'] * r['image']['height']
        if any(a['bbox'][2] * a['bbox'][3] / area > BLOB_AREA_RATIO for a in r['anns']):
            dropped['大框(多物件合併/特寫)'] += 1; continue
        kept.append(r)
    for k, v in dropped.items():
        print('    剔除 %s: %d 張' % (k, v))
    return kept

def _write(out_dir, categories, sources):
    os.makedirs(out_dir, exist_ok=True)
    images, anns, iid, aid, seen = [], [], 1, 1, set()
    for src_dir, records in sources:
        for r in records:
            fn = r['image']['file_name']
            if fn in seen:
                print('    [!] 檔名重複,跳過:', fn); continue
            seen.add(fn)
            images.append(dict(r['image'], id=iid))
            for a in r['anns']:
                anns.append(dict(a, id=aid, image_id=iid)); aid += 1
            iid += 1
            dst = os.path.join(out_dir, fn)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(src_dir, fn), dst)
    json.dump({'images': images, 'annotations': anns, 'categories': categories},
              open(os.path.join(out_dir, ANN), 'w', encoding='utf-8'), ensure_ascii=False)
    id2n = {c['id']: c['name'] for c in categories}
    cnt = Counter(id2n[a['category_id']] for a in anns)
    print('  %s: %d 圖 / %d 框 (平均 %.2f 框/圖)' % (
        os.path.basename(out_dir), len(images), len(anns), len(anns) / max(1, len(images))))
    print('    ' + '  '.join('%s=%d' % (k, v) for k, v in cnt.most_common()))

MIX = '/content/mix'
cats_e, tr_e = _load('/content/epfl/train')
cats_r, tr_r = _load('/content/rf/train')
assert [c['name'] for c in cats_e] == [c['name'] for c in cats_r], '類別清單不一致'
print('來源:EPFL train %d 圖 / Roboflow train %d 圖  (CLEAN_RF=%s)' % (len(tr_e), len(tr_r), CLEAN_RF))
if CLEAN_RF:
    tr_r = _filter_rf(tr_r)
    print('    保留 %d 張' % len(tr_r))

print('\n輸出 → /content/mix')
_write(MIX + '/train', cats_e, [('/content/epfl/train', tr_e), ('/content/rf/train', tr_r)])
for sp in ('valid', 'test'):
    cats, recs = _load('/content/epfl/' + sp)
    _write(MIX + '/' + sp, cats, [('/content/epfl/' + sp, recs)])
print('\n混合訓練集就緒')

In [ ]:
# 4) 評估工具(與 Arm A/B 逐字相同,確保分數可比)
# 3) 評估工具(同 m3_finetune 的 eval_model;test = EPFL test)
import os, json, gc
from collections import defaultdict
from PIL import Image
try:
    import torch
except Exception:
    torch = None

TEST = '/content/epfl/test'
NAMES = {0:'人',1:'刀具',2:'砧板',3:'食材',4:'鍋鏟',5:'鍋子',6:'手',7:'容器',8:'抹布',9:'夾子',10:'手套'}
CAT2CLS = {i: i-1 for i in range(1, 12)}

def free():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

def _load_gt(test_dir):
    j = json.load(open(os.path.join(test_dir, '_annotations.coco.json'), encoding='utf-8'))
    id2fn = {im['id']: im['file_name'] for im in j['images']}
    by = {}
    for a in j['annotations']:
        x, y, w, h = a['bbox']
        by.setdefault(id2fn[a['image_id']], []).append((a['category_id'], [x, y, x+w, y+h]))
    return by

def eval_model(model, test_dir, cat_to_class=None, thr=0.3, iou_thr=0.5):
    gt = _load_gt(test_dir)
    TP = defaultdict(int); FP = defaultdict(int); FN = defaultdict(int); NG = defaultdict(int)
    for fn, objs in gt.items():
        det = model.predict(Image.open(os.path.join(test_dir, fn)).convert('RGB'), threshold=thr)
        pboxes = det.xyxy.tolist() if len(det) else []
        pcls = [int(c) for c in det.class_id] if len(det) else []
        if len(det) and getattr(det, 'confidence', None) is not None:
            pconf = [float(x) for x in det.confidence]
        else:
            pconf = [1.0] * len(pboxes)
        gts = [(cat_to_class.get(gc, -1), gb) for gc, gb in objs]
        for gc, _ in gts:
            NG[gc] += 1
        mg = [False] * len(gts)
        for i in sorted(range(len(pboxes)), key=lambda k: -pconf[k]):
            c, box = pcls[i], pboxes[i]
            best, bestv = -1, iou_thr
            for j, (gc, gb) in enumerate(gts):
                if mg[j] or gc != c:
                    continue
                v = _iou(box, gb)
                if v >= bestv:
                    bestv, best = v, j
            if best >= 0:
                mg[best] = True; TP[c] += 1
            else:
                FP[c] += 1
        for j, (gc, gb) in enumerate(gts):
            if not mg[j]:
                FN[gc] += 1
    out = {'per_class': {}}
    tTP = tFP = tFN = 0
    for gc in sorted(NG):
        tp, fp, fn = TP[gc], FP[gc], FN[gc]
        tTP += tp; tFP += fp; tFN += fn
        p = tp/(tp+fp) if (tp+fp) else 0.0
        r = tp/(tp+fn) if (tp+fn) else 0.0
        f = 2*p*r/(p+r) if (p+r) else 0.0
        out['per_class'][NAMES.get(gc, gc)] = {'n_gt': NG[gc], 'precision': round(p,3), 'recall': round(r,3), 'f1': round(f,3)}
    P = tTP/(tTP+tFP) if (tTP+tFP) else 0.0
    R = tTP/(tTP+tFN) if (tTP+tFN) else 0.0
    F = 2*P*R/(P+R) if (P+R) else 0.0
    out['overall'] = {'precision': round(P,3), 'recall': round(R,3), 'f1': round(F,3)}
    return out

print('工具就緒。EPFL test 圖數:', len(_load_gt(TEST)))

In [ ]:
# 5) Arm C:混合訓練。設定與 Arm A / Arm B stage2 完全相同
from rfdetr import RFDETRNano
print('===== Arm C:EPFL + Roboflow 混合 =====')
mC = RFDETRNano()
mC.train(dataset_dir='/content/mix', epochs=60, batch_size=4, grad_accum_steps=4,
         lr=1e-4, resolution=704, output_dir='/content/out_C')
mC = None; free()

ftC = RFDETRNano(pretrain_weights='/content/out_C/checkpoint_best_regular.pth', num_classes=11)
resC = eval_model(ftC, TEST, cat_to_class=CAT2CLS)
print('Arm C overall:', resC['overall'])
ftC = None; free()

In [ ]:
# 6) 三臂比較(A / B 為 docs/M3_實驗_分階段vsEPFL_20260810.md 記錄值,未重訓)
import unicodedata

def pad(s, w):
    """中文全形字算 2 欄寬,讓表格對齊"""
    s = str(s)
    vis = sum(2 if unicodedata.east_asian_width(ch) in 'WF' else 1 for ch in s)
    return s + ' ' * max(0, w - vis)

REF = {
    'Arm A (EPFL-only)': {
        'overall': {'precision': 0.795, 'recall': 0.819, 'f1': 0.807},
        'per_class': {'刀具': {'precision': 0.773, 'recall': 0.785, 'f1': 0.779},
                      '容器': {'precision': 0.841, 'recall': 0.872, 'f1': 0.856}}},
    'Arm B (RF->EPFL 分階段)': {
        'overall': {'precision': 0.857, 'recall': 0.815, 'f1': 0.835},
        'per_class': {'刀具': {'precision': 0.836, 'recall': 0.785, 'f1': 0.810},
                      '容器': {'precision': 0.892, 'recall': 0.854, 'f1': 0.872}}},
}
ARMS = list(REF.items()) + [('Arm C (混合訓練)', resC)]

print('======== EPFL test 集(thr=0.3)三臂比較 ========\n')
print(pad('', 26) + '{:>9}{:>9}{:>9}'.format('Prec', 'Recall', 'F1'))
for name, res in ARMS:
    o = res['overall']
    print(pad(name, 26) + '{:>9.3f}{:>9.3f}{:>9.3f}'.format(o['precision'], o['recall'], o['f1']))

print('\n重點類別(Roboflow 唯一有標的兩類:刀具 / 容器)')
for cls in ['刀具', '容器']:
    print('\n  ' + cls)
    print('  ' + pad('', 26) + '{:>9}{:>9}{:>9}'.format('P', 'R', 'F1'))
    for name, res in ARMS:
        m = res['per_class'].get(cls, {})
        print('  ' + pad(name, 26) + '{:>9}{:>9}{:>9}'.format(
            m.get('precision', '-'), m.get('recall', '-'), m.get('f1', '-')))

print('\n漏標污染的受害類別(在 Roboflow 圖中出現但未標註)')
print('  ' + pad('類別', 10) + '{:>9}{:>9}{:>9}{:>10}'.format('P', 'R', 'F1', 'n_gt'))
for cls in ['砧板', '食材', '手', '人', '鍋子', '鍋鏟', '抹布']:
    m = resC['per_class'].get(cls, {})
    print('  ' + pad(cls, 10) + '{:>9}{:>9}{:>9}{:>10}'.format(
        m.get('precision', '-'), m.get('recall', '-'), m.get('f1', '-'), m.get('n_gt', '-')))

json.dump({'arm_C_mixed': resC, 'arm_A_B_reference': REF},
          open('/content/armC_results.json', 'w'), ensure_ascii=False, indent=2)
print('\n已存 /content/armC_results.json')
from google.colab import files
files.download('/content/armC_results.json')
try:
    files.download('/content/out_C/checkpoint_best_regular.pth')
except Exception as e:
    print('權重下載失敗:', e)

## 怎麼判讀

**主判準 —— 整體 F1**(A 0.807 / B 0.835):

| Arm C 結果 | 結論 |
|---|---|
| 明顯高於 B(> 0.86) | 混合確實優於分階段 —— 資料量的好處蓋過漏標的害處 |
| 落在 0.80~0.84 | 三臂全部打平,**結論不變:這份 Roboflow 資料沒用**,別再花時間 |
| 明顯低於 A(< 0.79) | **漏標污染成立**,混合有害。把 `CLEAN_RF = True` 再跑一次確認 |

**診斷判準 —— 砧板 / 食材 / 手**:
這三類在 Roboflow 圖中出現卻**完全沒有標註**,是漏標污染的直接受害者。
若它們的 recall 掉下來、而刀具/容器持平或微升 —— 那是典型的「用弱項換強項」,
**即使整體 F1 沒掉也不該採用**(砧板/食材/手正是食安判斷最需要的類)。

**⚠ 算力不對等**:C 的 60 epoch 跑 460 張(A 是 360 張),多約 28% 的 step 數。
若 C 只贏一點點,要先排除這是 step 數效果而非資料效果。

**⚠ test 只有 90 圖**,±2~3 F1 分在雜訊範圍內 —— 差距要夠大才算數。

把比較表 + `armC_results.json` 貼回來一起判讀。

## 相關
- Arm A / B 實驗:`docs/M3_實驗_分階段vsEPFL_20260810.md`
- Roboflow 資料集說明:`docs/資料集_Roboflow廚房_說明_20260820.md`
- 本機建同款資料集:`python scripts/build_mix_dataset.py [--clean-rf]`